In [ ]:
import requests

#авторизация на MOEX ISS через basic-аутентификацию
MOEX_LOGIN = 'orlovapolya0211@gmail.com' #логин от moex.com
MOEX_PASSWORD = 'E!3UStXK#wAwLgh'  #пароль от moex.com

session = requests.Session()
avtoriz_otvet = session.get(
    'https://passport.moex.com/authenticate',
    auth=(MOEX_LOGIN, MOEX_PASSWORD))

if avtoriz_otvet.status_code == 200:
    print('Авторизация успешна')
    print(f'Cookie получен: {"MicexPassportCert" in session.cookies}')
else:
    print(f'Ошибка авторизации: {avtoriz_otvet.status_code}')

Авторизация успешна
Cookie получен: True


In [ ]:
#cоздаём конфигурационный файл parametrs.txt
#параметры берём из статьи Tproger
#там написано - у basicConfig() три основных параметра - level, filename, format

parametrs_content = """enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
"""
#enabled=true/false - включение/выключение logging без изменения кода
#level=INFO
#filename=logs/logging_infa - где хранятся наши логи
#Tproger: format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
#убрали funcName потому что у нас нет функций
#%(asctime)s  - дата и время события
#%(levelname)s - уровень: INFO в нашем случае
#%(message)s  - само сообщение
#%(lineno)d - номер строки


with open('parametrs.txt', 'w') as filik:
    filik.write(parametrs_content)

print('создали parametrs.txt с такими параметрами:')
with open('parametrs.txt') as filik:
    print(filik.read())

создали parametrs.txt с такими параметрами:
enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s



In [ ]:
!mkdir -p logs #создали папку, где будет файл с логами

In [ ]:
import logging

#читаем наши настройки из parametrs.txt
with open('parametrs.txt') as filik:
    lines = filik.readlines()

enabled = lines[0].split('=')[1].strip()
level_str = lines[1].split('=')[1].strip()
filename = lines[2].split('=')[1].strip()
log_format = lines[3].split('=')[1].strip()

if enabled == 'true':

    #сбрасываем старые handlers
    #потому что иначе basicConfig игнорируется в Colab
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    #filename - из parametrs.txt
    #из Хабра нашли, что записываем сообщения в файл
    #файл будет хранить данные после завершения программы

    #format - из parametrs.txt
    #Tproger: format с %(asctime)s %(levelname)s %(message)s

    #запись в файл
    #Tproger: 'logging.basicConfig() - основная функция для настройки logging'
    #уровень INFO - вывод данных о фрагментах кода, работающих так как ожидается
    logging.basicConfig(level=logging.INFO, filename=filename, format=log_format, filemode='a')
    #filemode='a' - логи добавляются в конец файла и не стираются при перезапуске

    logging.info('Подключаем logging и собираем данные с MOEX ISS API')
    print(f'Logging работает: уровень={level_str}, файл={filename}')

else:
    print('Logging не работает')

Logging работает: уровень=INFO, файл=logs/logging_infa


In [ ]:
import requests #сначала все запросы писали через requests.get() напрямую, потом переписали через session для логгирования
import pandas as pd

Импортируем две библиотеки:
- **requests** - для отправки HTTP-запросов к API биржи (для колаба без логгирования нужна была)
- **pandas** - для работы с таблицами. Превращаем неудобный JSON в удобный DataFrame

## Запрос 1 - Дневные свечи (candles)

**Цель:** получить историю дневных цен Норникеля за период 2018-01-01 - 2025-12-31.

Это главные данные всего проекта. Без них невозможно посчитать нашу таргетную переменную - изменение цены акции через 24 часа после выхода новости.

Каждая строка в ответе = один торговый день.


In [ ]:
#запрос 1: получаем дневные свечи НОРНИКЕЛЬ с MOEX (первые 500 строк)
response_GMKN = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GMKN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24})

data_GMKN = response_GMKN.json()

#извлекаем данные из json
columns_GMKN = data_GMKN['candles']['columns']
rows_GMKN = data_GMKN['candles']['data']

df_GMKN = pd.DataFrame(rows_GMKN, columns=columns_GMKN)
df_GMKN['ticker'] = 'GMKN'

print(f'Count of rows: {len(df_GMKN)}')
#считаем все строки - мало ли их много, а выводит только 500
df_GMKN


Count of rows: 500


,open,close,high,low,value,volume,begin,end,ticker
0,108.80,113.02,113.02,108.80,1697344891,152990,2018-01-03 00:00:00,2018-01-03 23:59:59,GMKN
1,113.10,115.85,116.00,112.69,2313208233,200891,2018-01-04 00:00:00,2018-01-04 23:59:59,GMKN
2,116.06,115.65,116.39,114.56,2208975336,191238,2018-01-05 00:00:00,2018-01-05 23:59:59,GMKN
3,116.10,116.75,117.94,115.90,2866064107,244866,2018-01-09 00:00:00,2018-01-09 23:59:59,GMKN
4,116.70,116.55,116.97,114.82,2684281888,231328,2018-01-10 00:00:00,2018-01-10 23:59:59,GMKN
...,...,...,...,...,...,...,...,...,...
495,195.00,197.22,199.78,194.18,6318827714,319268,2019-12-16 00:00:00,2019-12-16 23:59:59,GMKN
496,197.50,195.04,197.96,194.12,5687095198,290111,2019-12-17 00:00:00,2019-12-17 23:59:59,GMKN
497,195.00,196.26,196.26,192.22,5350999132,275874,2019-12-18 00:00:00,2019-12-18 23:59:59,GMKN
498,196.10,197.96,198.50,196.00,4146845472,209622,2019-12-19 00:00:00,2019-12-19 23:59:59,GMKN


**Результат:**  Count of rows: 500

Таблица содержит 9 колонок:
-  open  - цена акции Норникеля в начале торгового дня (рублей за акцию)
-  close  - цена в конце торгового дня
-  high  - максимальная цена за день
-  low  - минимальная цена за день
-  value  - суммарный оборот за день в рублях (сколько рублей прошло через все сделки)
-  volume  - количество акций сменивших владельца за день
-  begin  - дата и время начала торгового дня
-  end  - дата и время конца торгового дня
-  ticker  - тикер бумаги (добавили сами, чтобы потом объединить все 7 компаний)


Также из документации (стр. 11) узнали что сервер возвращает данные порциями. Чтобы выгрузить полный объём данных, нужно последовательно делать запросы, увеличивая параметр start на значение pagesize, пока не получим пустой блок данных.



In [ ]:
logging.info('Запрос 1: проверяем есть ли данные после 500 строки (start=500)')
#проверяем есть ли данные ниже 500 строки
response_GMKN = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GMKN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 500
      })

data_GMKN = response_GMKN.json()
print(f"Count of rows after 500 rows: {len(data_GMKN['candles']['data'])}")
#строк много - выводим оставшиеся


Count of rows after 500 rows: 500


**Результат:**  Count of rows after 500 rows: 500

HOOORRAY!!))

Ну вот видите?)

Данные не закончились на 500-й строке. Получили ещё 500 строк - это оставшиеся торговые дни, которые не влезли в первый запрос.

Теперь будем проверять, пока нам не выведет 0 значений.

In [ ]:
columns_GMKNnew = data_GMKN['candles']['columns']
rows_GMKNnew = data_GMKN['candles']['data']

df_GMKNnew = pd.DataFrame(rows_GMKNnew, columns=columns_GMKNnew)
df_GMKNnew['ticker'] = 'GMKN'

print(f'Count of rows after 500 rows: {len(df_GMKNnew)}')

#проверяем есть ли данные ниже 1000 строки
logging.info('Запрос 1: проверяем есть ли данные после 1000 строки (start=1000)')
response_GMKN2 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GMKN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1000
      })

data_GMKN2 = response_GMKN2.json()
columns_GMKNnew2 = data_GMKN2['candles']['columns']
rows_GMKNnew2 = data_GMKN2['candles']['data']

df_GMKNnew2 = pd.DataFrame(rows_GMKNnew2, columns=columns_GMKNnew2)
df_GMKNnew2['ticker'] = 'GMKN'

print(f'Count of rows after 1000 rows: {len(df_GMKNnew2)}')

#проверяем есть ли данные ниже 1500 строки
logging.info('Запрос 1: проверяем есть ли данные после 1500 строки (start=1500)')
response_GMKN3 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GMKN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1500
      })

data_GMKN3 = response_GMKN3.json()
columns_GMKNnew3 = data_GMKN3['candles']['columns']
rows_GMKNnew3 = data_GMKN3['candles']['data']

df_GMKNnew3 = pd.DataFrame(rows_GMKNnew3, columns=columns_GMKNnew3)
df_GMKNnew3['ticker'] = 'GMKN'

print(f'Count of rows after 1500 rows: {len(df_GMKNnew3)}')



Count of rows after 500 rows: 500
Count of rows after 1000 rows: 500
Count of rows after 1500 rows: 500


In [ ]:
#проверяем есть ли данные ниже 2000 строки

logging.info('Запрос 1: проверяем есть ли данные после 2000 строки (start=2000)')
response_GMKN4 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GMKN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2000
      })

data_GMKN4 = response_GMKN4.json()
columns_GMKNnew4 = data_GMKN4['candles']['columns']
rows_GMKNnew4 = data_GMKN4['candles']['data']

df_GMKNnew4 = pd.DataFrame(rows_GMKNnew4, columns=columns_GMKNnew4)
df_GMKNnew4['ticker'] = 'GMKN'

print(f'Count of rows after 2000 rows: {len(df_GMKNnew4)}')

Count of rows after 2000 rows: 69


In [ ]:
#проверяем есть ли данные ниже 2069 строки
logging.info('Запрос 1: проверяем есть ли данные после 2069 строки (start=2069)')
response_GMKN5 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GMKN/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2069
      })

data_GMKN5 = response_GMKN5.json()
columns_GMKNnew5 = data_GMKN5['candles']['columns']
rows_GMKNnew5 = data_GMKN5['candles']['data']

df_GMKNnew5 = pd.DataFrame(rows_GMKNnew5, columns=columns_GMKNnew5)
df_GMKNnew5['ticker'] = 'GMKN'

print(f'Count of rows after 2069 rows: {len(df_GMKNnew5)}')

Count of rows after 2069 rows: 0


In [ ]:
#больше данных нет, поэтому обьединяем датафреймы в один
df_GMKNcandles = pd.concat([df_GMKN, df_GMKNnew, df_GMKNnew2, df_GMKNnew3, df_GMKNnew4], ignore_index=True)
print(f'Total rows: {len(df_GMKNcandles)}')

logging.info(f'Запрос 1: итого после склейки пяти датафреймов - {len(df_GMKNcandles)} строк')
df_GMKNcandles

Total rows: 2069


,open,close,high,low,value,volume,begin,end,ticker
0,108.80,113.02,113.02,108.80,1.697345e+09,152990,2018-01-03 00:00:00,2018-01-03 23:59:59,GMKN
1,113.10,115.85,116.00,112.69,2.313208e+09,200891,2018-01-04 00:00:00,2018-01-04 23:59:59,GMKN
2,116.06,115.65,116.39,114.56,2.208975e+09,191238,2018-01-05 00:00:00,2018-01-05 23:59:59,GMKN
3,116.10,116.75,117.94,115.90,2.866064e+09,244866,2018-01-09 00:00:00,2018-01-09 23:59:59,GMKN
4,116.70,116.55,116.97,114.82,2.684282e+09,231328,2018-01-10 00:00:00,2018-01-10 23:59:59,GMKN
...,...,...,...,...,...,...,...,...,...
2064,147.40,152.00,152.00,147.30,4.677089e+09,31280500,2025-12-26 00:00:00,2025-12-26 23:59:58,GMKN
2065,152.26,152.56,152.76,151.94,5.326925e+08,3495290,2025-12-27 00:00:00,2025-12-27 23:59:57,GMKN
2066,152.70,152.72,152.98,151.30,5.038317e+08,3308550,2025-12-28 00:00:00,2025-12-28 23:59:57,GMKN
2067,152.68,148.70,153.10,148.08,5.324781e+09,35374120,2025-12-29 00:00:00,2025-12-29 23:59:59,GMKN


Hooray!

Пустой ответ подтверждает, что мы выгрузили абсолютно все данные. После 2069 строки данных нет. Значит итоговый датафрейм  df_GMKNcandles  содержит полную историю дневных котировок Норникеля за весь нужный период без пропусков.

**Результат:**
   
Total rows: 2069
   

Склеили пять кусков через  pd.concat  с параметром  ignore_index=True  - пересчитаем индексы строк заново от 0, иначе в итоговой таблице было бы пять наборов индексов, что вызвало бы у нас путаницу и отсутствие hoorray(.

Итого получили **2069 строк** - это все торговые дни Норникеля за период с 01.01.2018 по 31.12.2025.



## Запрос 2 - Информация о бумаге (security info)

**Цель:** получить полную информацию по акциям Норникеля: официальное название компании, международный код бумаги (ISIN), номинальная стоимость акции и уровень листинга.

Эти данные нужны нам для двух вещей:
1. Красиво описать датасет в README - чтобы было понятно с какими именно бумагами работаем
2. Добавить уровень листинга как признак в EDA - бумаги первого уровня листинга торгуются активнее и имеют более строгие требования к раскрытию информации

In [ ]:
#запрос 2: получаем статические характеристики бумаги НОРНИКЕЛЬ
#нам возвращается: полное название, ISIN, номинал, уровень листинга, тип бумаги
response_GMKN_infa = session.get(
    'https://iss.moex.com/iss/securities/GMKN.json')

data_GMKN_infa = response_GMKN_infa.json()

#раздел 'description' содержит характеристики в виде строк name/value
columns_GMKN_infa = data_GMKN_infa['description']['columns']
rows_GMKN_infa = data_GMKN_infa['description']['data']

df_GMKN_infa = pd.DataFrame(rows_GMKN_infa, columns=columns_GMKN_infa)
df_GMKN_infa['ticker'] = 'GMKN'

print(f'Count of rows: {len(df_GMKN_infa)}')
df_GMKN_infa

Count of rows: 27


,name,title,value,type,sort_order,is_hidden,precision,ticker
0,SECID,Код ценной бумаги,GMKN,string,1,0,NaN,GMKN
1,ISSUENAME,Наименование ценной бумаги,акция обыкновенная бездокументарная именная,string,2,0,NaN,GMKN
2,NAME,Полное наименование,"ГМК ""Нор.Никель"" ПАО ао",string,3,0,NaN,GMKN
3,SHORTNAME,Краткое наименование,ГМКНорНик,string,4,0,NaN,GMKN
4,ISIN,ISIN код,RU0007288411,string,5,0,NaN,GMKN
5,REGNUMBER,Номер государственной регистрации,1-01-40155-F,string,6,0,NaN,GMKN
6,ISSUESIZE,Объем выпуска,15286339700,number,7,0,0.0,GMKN
7,FACEVALUE,Номинальная стоимость,0.01,number,8,0,2.0,GMKN
8,FACEUNIT,Валюта номинала,SUR,string,9,0,NaN,GMKN
9,ISSUEDATE,Дата начала торгов,2006-12-26,date,10,0,NaN,GMKN


**Результат:**  Count of rows: 27

Получили 27 строк.

Каждая строка это одна характеристика бумаги.

Каждая характеристика на отдельной строке.

Это неудобно для анализа, поэтому в следующей ячейке мы преобразуем это формат, где одна строка на компанию, каждая характеристика отдельная колонка.

In [ ]:
#преобразуем в удобный формат: одна строка - один тикер
#разворачиваем колонку 'name' в отдельные колонки через pivot
df_GMKN_infa_best = df_GMKN_infa.set_index('name')['value'].to_frame().T
df_GMKN_infa_best['ticker'] = 'GMKN'
df_GMKN_infa_best = df_GMKN_infa_best.reset_index()

print(f'Полное название: {df_GMKN_infa_best["NAME"].values[0]}')
print(f'ISIN: {df_GMKN_infa_best["ISIN"].values[0]}')
print(f'Уровень листинга: {df_GMKN_infa_best["LISTLEVEL"].values[0]}')

logging.info(f'Запрос 2: ISIN={df_GMKN_infa_best["ISIN"].values[0]}, листинг={df_GMKN_infa_best["LISTLEVEL"].values[0]}')

df_GMKN_infa_best

Полное название: ГМК "Нор.Никель" ПАО ао
ISIN: RU0007288411
Уровень листинга: 1


name,index,SECID,ISSUENAME,NAME,SHORTNAME,ISIN,REGNUMBER,ISSUESIZE,FACEVALUE,FACEUNIT,...,MORNINGSESSION,EVENINGSESSION,WEEKENDSESSION,REGISTRY_DATE,TYPENAME,GROUP,TYPE,GROUPNAME,EMITTER_ID,ticker
0,value,GMKN,акция обыкновенная бездокументарная именная,"ГМК ""Нор.Никель"" ПАО ао",ГМКНорНик,RU0007288411,1-01-40155-F,15286339700,0.01,SUR,...,1,1,1,2006-12-12,Акция обыкновенная,stock_shares,common_share,Акции,1399,GMKN


**Результат:**
   
Полное название: ГМК "Нор.Никель" ПАО ао
ISIN: RU0007288411
Уровень листинга: 1
   

Теперь это одна строка с колонками NAME, ISIN, LISTLEVEL и тд.

Что значат выводы:
- **NAME = «ГМК "Нор.Никель" ПАО ао»** - официальное юридическое название компании.
- **ISIN = RU0007288411** - международный идентификационный номер ценной бумаги.
- **LISTLEVEL = 1** - первый уровень листинга на Мосбирже. Это значит, что Норникель прошёл самые строгие требования биржи по прозрачности и раскрытию информации.

**Итог запроса 2:** получили официальную информацию по бумаге.

Hooray!))

## Запрос 3 - История дивидендов (dividends)

**Цель:** получить все даты дивидендных выплат Норникеля за период проекта.

Зачем нам это нужно: когда компания выплачивает дивиденды, цена акции резко падает примерно на размер дивиденда.

Если мы не пометим эти дни, то можем ошибочно решить в EDA, что резкое падение цены вызвано негативной новостью - хотя на самом деле это просто плановая выплата. Это могло бы нам помешать.

In [ ]:
logging.info('Запрос 3: загружаем историю дивидендов GMKN')
#запрос 3: получаем историю дивидендных выплат НОРНИКЕЛЬ
#дивиденды влияют на котировки примерно на размер дивиденда
GMKN_diva = session.get(
    'https://iss.moex.com/iss/securities/GMKN/dividends.json')

GMKN_diva_infa = GMKN_diva.json()

#извлекаем данные из json
columns_GMKN_diva = GMKN_diva_infa['dividends']['columns']
rows_GMKN_diva = GMKN_diva_infa['dividends']['data']

df_GMKN_diva = pd.DataFrame(rows_GMKN_diva, columns=columns_GMKN_diva)
df_GMKN_diva['ticker'] = 'GMKN'

print(f'Count of rows: {len(df_GMKN_diva)}')

logging.info(f'Запрос 3: получено {len(df_GMKN_diva)} дивидендных выплат')
df_GMKN_diva

Count of rows: 21


,secid,isin,registryclosedate,value,currencyid,ticker
0,GMKN,RU0007288411,2013-11-01,220.70,RUB,GMKN
1,GMKN,RU0007288411,2014-06-17,248.48,RUB,GMKN
2,GMKN,RU0007288411,2014-12-22,762.34,RUB,GMKN
3,GMKN,RU0007288411,2015-05-25,670.04,RUB,GMKN
4,GMKN,RU0007288411,2015-09-25,305.07,RUB,GMKN
5,GMKN,RU0007288411,2015-12-30,321.95,RUB,GMKN
6,GMKN,RU0007288411,2016-06-21,230.14,RUB,GMKN
7,GMKN,RU0007288411,2016-12-28,444.25,RUB,GMKN
8,GMKN,RU0007288411,2017-06-23,446.10,RUB,GMKN
9,GMKN,RU0007288411,2017-10-19,224.20,RUB,GMKN


**Результат:**  Count of rows: 21

API вернул 21 выплат дивидентов за все годы существования на бирже. Таблица содержит колонки:

- **registryclosedate** - дата закрытия реестра.
- **value** - размер дивиденда на одну акцию в рублях.

Нам нужны только выплаты за наш период 2018–2025, поэтому сейчас будет фильтровать.

In [ ]:
#фильтруем дивиденды по диапазону дат проекта 2018-01-01 - 2025-12-31
df_GMKN_diva['registryclosedate'] = pd.to_datetime(df_GMKN_diva['registryclosedate'])
df_GMKN_diva_srok = df_GMKN_diva[
    (df_GMKN_diva['registryclosedate'] >= '2018-01-01') &
    (df_GMKN_diva['registryclosedate'] <= '2025-12-31')].reset_index()

print(f'Все дивиденты за 2018-01-01 - 2025-12-31: {len(df_GMKN_diva_srok)}')

logging.info(f'Запрос 2: ISIN={df_GMKN_infa_best["ISIN"].values[0]}, листинг={df_GMKN_infa_best["LISTLEVEL"].values[0]}')
df_GMKN_diva_srok

Все дивиденты за 2018-01-01 - 2025-12-31: 11


,index,secid,isin,registryclosedate,value,currencyid,ticker
0,10,GMKN,RU0007288411,2018-07-17,607.98,RUB,GMKN
1,11,GMKN,RU0007288411,2018-10-01,776.02,RUB,GMKN
2,12,GMKN,RU0007288411,2019-06-21,792.52,RUB,GMKN
3,13,GMKN,RU0007288411,2019-10-07,883.93,RUB,GMKN
4,14,GMKN,RU0007288411,2019-12-27,604.09,RUB,GMKN
5,15,GMKN,RU0007288411,2020-05-25,557.20,RUB,GMKN
6,16,GMKN,RU0007288411,2020-12-24,623.35,RUB,GMKN
7,17,GMKN,RU0007288411,2021-06-01,1021.22,RUB,GMKN
8,18,GMKN,RU0007288411,2022-01-14,1523.17,RUB,GMKN
9,19,GMKN,RU0007288411,2022-06-14,1166.22,RUB,GMKN


**Результат:**  Все дивиденты за 2018-01-01 - 2025-12-31: 11


Почему конвертировали дату через  pd.to_datetime : без этого Python сравнивал бы строки а не даты. Строковое сравнение работает по алфавиту и даёт неверные результаты для дат в формате YYYY-MM-DD.

**Итог запроса 3:** знаем точные даты дивидендных выплат. В EDA пометим эти дни флагом  is_dividend_day=1 .

## Запрос 4 - Текущие рыночные данные (market data)

Запрос 4 получает текущие торговые данные по Норникелю - последнюю цену, объём торгов за день, количество сделок и капитализацию компании. Всё это показывает насколько активно торгуется акция прямо сейчас.


Эти данные характеризуют **ликвидность** бумаги - насколько активно она торгуется. В EDA мы будем использовать их чтобы показать: более ликвидные бумаги быстрее и точнее реагируют на новости, потому что больше участников рынка мгновенно перекладываются после публикации статьи.

In [ ]:
logging.info('Запрос 4: загружаем текущие рыночные данные GMKN')
#запрос 4: получаем текущие торговые данные НОРНИКЕЛЬ
response_GMKN_markdata = session.get(
    'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/GMKN.json')

data_GMKN_markdata = response_GMKN_markdata.json()


#раздел 'marketdata' содержит торговые данные текущей/последней сессии
columns_GMKN_markdata = data_GMKN_markdata['marketdata']['columns']
rows_GMKN_markdata = data_GMKN_markdata['marketdata']['data']

df_GMKN_markdata = pd.DataFrame(rows_GMKN_markdata, columns=columns_GMKN_markdata)
df_GMKN_markdata['ticker'] = 'GMKN'

print(f'Count of rows: {len(df_GMKN_markdata)}')

logging.info(f'Запрос 4: получено {len(df_GMKN_markdata)} строк market data')
df_GMKN_markdata

Count of rows: 1


,SECID,BOARDID,BID,BIDDEPTH,OFFER,OFFERDEPTH,SPREAD,BIDDEPTHT,OFFERDEPTHT,OPEN,...,SYSTIME,CLOSINGAUCTIONPRICE,CLOSINGAUCTIONVOLUME,ISSUECAPITALIZATION,ISSUECAPITALIZATION_UPDATETIME,ETFSETTLECURRENCY,VALTODAY_RUR,TRADINGSESSION,TRENDISSUECAPITALIZATION,ticker
0,GMKN,TQBR,152.8,None,152.86,None,0.06,117937,198641,152.16,...,2026-03-19 23:53:08,151,35850,2331472531044,23:49:58,None,2663636765,2,5808809086,GMKN


**Результат:**  Count of rows: 1

Одна строка - потому что это текущие торговые данные. Но колонок очень много. API возвращает абсолютно всё что знает о торговле этой бумагой прямо сейчас.

Большинство колонок нам не нужны - это технические поля биржи вроде кодов режимов торгов, флагов состояния и прочего. В следующей ячейке оставим только то что важно для нашего анализа.

In [ ]:
#оставляем только нужные колонки из market data
#используем фильтрацию через список - так код не упадёт если какой-то колонки нет в ответе
markdata_cols = ['ticker', 'LAST', 'LASTTOPREVPRICE', 'BID', 'OFFER',
            'OPEN', 'HIGH', 'LOW', 'WAPRICE', 'NUMTRADES',
            'VOLTODAY', 'VALTODAY', 'ISSUECAPITALIZATION', 'UPDATETIME']
markdata_cols = [col for col in markdata_cols if col in df_GMKN_markdata.columns]

df_GMKN_markdata_best = df_GMKN_markdata[markdata_cols].copy()

#выводим ключевые показатели
print(f'Последняя цена GMKN: {df_GMKN_markdata_best["LAST"].values[0]}')
print(f'Изменение к пред. закрытию: {df_GMKN_markdata_best["LASTTOPREVPRICE"].values[0]}')
print(f'Капитализация: {df_GMKN_markdata_best["ISSUECAPITALIZATION"].values[0]}')

logging.info(f'Запрос 4: последняя цена GMKN={df_GMKN_markdata_best["LAST"].values[0]}')

df_GMKN_markdata_best

Последняя цена GMKN: 152.86
Изменение к пред. закрытию: 0.47
Капитализация: 2331472531044


,ticker,LAST,LASTTOPREVPRICE,BID,OFFER,OPEN,HIGH,LOW,WAPRICE,NUMTRADES,VOLTODAY,VALTODAY,ISSUECAPITALIZATION,UPDATETIME
0,GMKN,152.86,0.47,152.8,152.86,152.16,153.5,150.8,152.14,49473,17510130,2663636765,2331472531044,23:38:07


**Результат:** (тут данные могут изменяться при каждом запуске)
   
Последняя цена GMKN: 152.84
Изменение к пред. закрытию: 0.46
Капитализация: 2335446979366
   

Что означают колонки которые мы оставили:
- **LAST** - последняя цена сделки (рублей за акцию)
- **LASTTOPREVPRICE** - изменение в % к цене закрытия предыдущего дня
- **BID / OFFER** - лучшая цена покупки и продажи прямо сейчас. Разница между ними называется спредом - чем он уже, тем ликвиднее бумага
- **WAPRICE** - средневзвешенная цена за весь день (учитывает объём каждой сделки)
- **NUMTRADES** - количество сделок за торговый день. У Норникеля это десятки тысяч - одна из самых торгуемых бумаг на бирже
- **VOLTODAY / VALTODAY** - объём в акциях и оборот в рублях за сегодня
- **ISSUECAPITALIZATION** - рыночная капитализация в рублях.

**Итог запроса 4:** получили характеристику ликвидности Норникеля на текущий момент.

## Запрос 5 - Индексная принадлежность (indices)

**Цель:** узнать в какие биржевые индексы входит Норникель.

Индексные бумаги - особая категория. Когда акция входит в IMOEX (Индекс МосБиржи), за ней автоматически следят индексные фонды (ETF) и управляющие компании. Это означает что при выходе важной новости реагируют не только частные инвесторы, но и алгоритмы фондов.

В EDA мы сравним: одинаково ли сильно реагируют на новости индексные бумаги и менее крупные компании из нашего списка.

In [ ]:
logging.info('Запрос 5: загружаем индексы GMKN')
#запрос 5: получаем список биржевых индексов в которые входит НОРНИКЕЛЬ
response_GMKN_idishka = session.get(
    'https://iss.moex.com/iss/securities/GMKN/indices.json')

data_GMKN_idishka = response_GMKN_idishka.json()

#извлекаем данные из json
columns_GMKN_idishka = data_GMKN_idishka['indices']['columns']
rows_GMKN_idishka = data_GMKN_idishka['indices']['data']

df_GMKN_idishka = pd.DataFrame(rows_GMKN_idishka, columns=columns_GMKN_idishka)
df_GMKN_idishka['ticker'] = 'GMKN'

print(f'Count of rows: {len(df_GMKN_idishka)}')

logging.info(f'Запрос 5: GMKN входит в {len(df_GMKN_idishka)} индексов')

df_GMKN_idishka

Count of rows: 23


,SECID,SHORTNAME,FROM,TILL,CURRENTVALUE,LASTCHANGEPRC,LASTCHANGE,ticker
0,EPSI,Субиндекс акций,2010-06-15,2026-03-18,1665.38,0.00,-0.02,GMKN
1,ESGI,Индекс МосБиржи Рейтинги УР,2026-02-16,2026-03-18,988.87,-0.96,-9.62,GMKN
2,HCAP,HCAP,2024-06-21,2024-09-26,907.67,-0.07,-0.66,GMKN
3,ICLIMATE,Индекс МосБиржи Климатический,2025-07-01,2026-03-18,1030.83,0.23,2.39,GMKN
4,IMOEXCNY,Индекс МосБиржи в юанях,2024-09-24,2026-03-18,1060.38,-2.49,-27.11,GMKN
5,IMOEXW,IMOEXW,2023-01-16,2026-03-18,2881.58,-0.15,-4.34,GMKN
6,MESG,Индекс МосБиржи-RAEX ESG,2023-02-27,2026-03-18,984.75,-0.27,-2.69,GMKN
7,MOEXBC,Индекс голубых фишек,2010-01-14,2026-03-19,19057.56,-0.03,-5.43,GMKN
8,MRRT,Индекс MRRT,2020-09-21,2026-03-18,2335.80,0.05,1.15,GMKN
9,MRSV,Индекс MRSV,2020-09-21,2026-03-18,2173.86,-0.22,-4.80,GMKN


**Результат:**  Count of rows: 23


Важные колонки:
- **SECID** - код индекса
- **FROM / TILL** - период с которого по который бумага входит в индекс
- **CURRENTVALUE** - текущее значение самого индекса

In [ ]:
#проверяем входит ли GMKN в IMOEX (главный российский фондовый индекс)
if len(df_GMKN_idishka) > 0:
    imoex_check = df_GMKN_idishka[df_GMKN_idishka['SECID'] == 'IMOEX']
    print(f'GMKN входит в IMOEX: {len(imoex_check) > 0}')
else:
    print('GMKN не входит ни в один индекс MOEX')
logging.info(f'Запрос 5 завершили - все данные по GMKN собраны')


GMKN входит в IMOEX: True


**Результат:**
   
GMKN входит в IMOEX: True
   

Норникель входит в IMOEX - это подтверждает что он является влиятельной компанией российского рынка.

**Итог запроса 5:** Норникель входит в 23 индексов включая главный IMOE. В EDA это будет важным признаком при анализе силы реакции на новости.



